## Driver Drowsiness Detection — EfficientNet-B0 (Transfer Learning)
Same 6-class task as the baseline CNN notebook, but using EfficientNet-B0 pretrained on ImageNet.

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt

# labels

In [ ]:
labels = os.listdir("../input/drowsiness-dataset/train")
print(labels)

# IMG_SIZE = 224 for EfficientNet-B0 (designed for 224x224 input)

In [ ]:
IMG_SIZE = 224

# for yawn and no_yawn — detect and crop face region

In [ ]:
def face_for_yawn(direc="../input/drowsiness-dataset/train", face_cas_path="../input/prediction-images/haarcascade_frontalface_default.xml"):
    yaw_no = []
    categories = ["yawn", "no_yawn"]
    for category in categories:
        path_link = os.path.join(direc, category)
        class_num1 = categories.index(category)
        print(class_num1)
        for image in os.listdir(path_link):
            image_array = cv2.imread(os.path.join(path_link, image), cv2.IMREAD_COLOR)
            face_cascade = cv2.CascadeClassifier(face_cas_path)
            faces = face_cascade.detectMultiScale(image_array, 1.3, 5)
            for (x, y, w, h) in faces:
                img = cv2.rectangle(image_array, (x, y), (x+w, y+h), (0, 255, 0), 2)
                roi_color = img[y:y+h, x:x+w]
                resized_array = cv2.resize(roi_color, (IMG_SIZE, IMG_SIZE))
                yaw_no.append([resized_array, class_num1])
    return yaw_no


yawn_no_yawn = face_for_yawn()

# for closed and open eye

In [ ]:
def get_data(dir_path="../input/drowsiness-dataset/train/", face_cas="../input/prediction-images/haarcascade_frontalface_default.xml", eye_cas="../input/prediction-images/haarcascade.xml"):
    labels = ['Closed', 'Open']
    data = []
    for label in labels:
        path = os.path.join(dir_path, label)
        class_num = labels.index(label)
        class_num += 2
        print(class_num)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_COLOR)
                resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                data.append([resized_array, class_num])
            except Exception as e:
                print(e)
    return data


data_train = get_data()

# for head pose (front and down)

In [ ]:
def get_head_pose_data(dir_path="/kaggle/input/datasets/antuchowdhury112/headpose/train"):
    categories = ["front", "down"]
    head_data = []
    for category in categories:
        path = os.path.join(dir_path, category)
        class_num = categories.index(category) + 4
        print(class_num)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_COLOR)
                if img_array is None:
                    continue
                resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                if resized_array.shape != (IMG_SIZE, IMG_SIZE, 3):
                    continue
                head_data.append([resized_array, class_num])
            except Exception as e:
                print(e)
    return head_data

# extend data and convert array

In [ ]:
def append_data():
    yaw_no = face_for_yawn()
    data = get_data()
    head_data = get_head_pose_data()
    yaw_no.extend(data)
    yaw_no.extend(head_data)
    features = np.array([item[0] for item in yaw_no])
    labels = np.array([item[1] for item in yaw_no])
    return list(zip(features, labels))

# new variable to store

In [ ]:
new_data = append_data()

# separate label and features

In [ ]:
X = []
y = []
for feature, label in new_data:
    X.append(feature)
    y.append(label)

# reshape the array

In [ ]:
X = np.array(X)
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 3)

# LabelBinarizer

In [ ]:
from sklearn.preprocessing import LabelBinarizer
label_bin = LabelBinarizer()
y = label_bin.fit_transform(y)

# label array

In [ ]:
y = np.array(y)

# train test split

In [ ]:
from sklearn.model_selection import train_test_split
seed = 42
test_size = 0.30
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=seed, test_size=test_size)

# length of X_test

In [ ]:
len(X_test)

# import dependencies

In [ ]:
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

# Data Augmentation
# Note: no rescale here — EfficientNetB0 includes its own internal normalization

In [ ]:
train_generator = ImageDataGenerator(zoom_range=0.2, horizontal_flip=True, rotation_range=30)
test_generator  = ImageDataGenerator()

train_generator = train_generator.flow(np.array(X_train), y_train, shuffle=False)
test_generator  = test_generator.flow(np.array(X_test), y_test, shuffle=False)

# Model — EfficientNet-B0 (Transfer Learning)
## Phase 1: freeze the entire base model, train only the new head

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(6, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer=Adam(learning_rate=1e-3)
)

model.summary()

# Phase 1 Training — head only

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

history1 = model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

# Phase 2 — Fine-tuning
## Unfreeze the last 30 layers of the base model and train with a very low learning rate

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer=Adam(learning_rate=1e-5)
)

early_stop2 = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr2  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

history2 = model.fit(
    train_generator,
    epochs=40,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    shuffle=True,
    callbacks=[early_stop2, reduce_lr2]
)

# Training history — combined Phase 1 + Phase 2

In [ ]:
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']
epochs  = range(len(acc))

plt.plot(epochs, acc,     'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.axvline(x=len(history1.history['accuracy']), color='gray', linestyle='--', label='Fine-tune start')
plt.legend()
plt.title('Accuracy')
plt.show()

plt.plot(epochs, loss,     'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.axvline(x=len(history1.history['loss']), color='gray', linestyle='--', label='Fine-tune start')
plt.legend()
plt.title('Loss')
plt.show()

# save model

In [ ]:
model.save('drowsiness_efficientnet_b0.h5')
model.save('drowsiness_efficientnet_b0.keras')

# Prediction

In [ ]:
prediction = np.argmax(model.predict(X_test), axis=1)

# classification report

In [ ]:
labels_new = ['yawn', 'no_yawn', 'Closed', 'Open', 'front', 'down']

from sklearn.metrics import classification_report
print(classification_report(np.argmax(y_test, axis=1), prediction, target_names=labels_new))

# predicting function

In [ ]:
labels_new = ['yawn', 'no_yawn', 'Closed', 'Open', 'front', 'down']

def prepare(filepath):
    img_array = cv2.imread(filepath, cv2.IMREAD_COLOR)
    resized   = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
    return resized.reshape(-1, IMG_SIZE, IMG_SIZE, 3)

model = tf.keras.models.load_model('drowsiness_efficientnet_b0.h5')

# Prediction
## 0-yawn, 1-no_yawn, 2-Closed, 3-Open, 4-front, 5-down

In [ ]:
prediction = model.predict([prepare('../input/drowsiness-dataset/train/no_yawn/1067.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('../input/drowsiness-dataset/train/Closed/_101.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('../input/drowsiness-dataset/train/Open/_104.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('../input/drowsiness-dataset/train/yawn/113.jpg')])
np.argmax(prediction)